In [ ]:
import os
import re
import time
from getpass import getpass
from pathlib import Path
from typing import Dict, List, Optional

import pandas as pd

In [ ]:
RUN_COLLECTION = False
FETCH_ABSTRACTS = True
ROOT = Path.cwd()
DATA_FILES = ['patents_script1_.csv', 'patents_script2_.csv']
DATA_DIR = next(
    (directory for directory in [ROOT / 'data', ROOT.parent / 'data', ROOT]
     if all((directory / name).is_file() for name in DATA_FILES)),
    None
)
if DATA_DIR is None and not RUN_COLLECTION:
    raise FileNotFoundError(', '.join(DATA_FILES))
PROJECT_DIR = DATA_DIR.parent if DATA_DIR is not None and DATA_DIR.name == 'data' else ROOT
RESULTS_DIR = PROJECT_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
COLLECTION_DIR = RESULTS_DIR
API_KEY = ''
if RUN_COLLECTION:
    API_KEY = os.environ.get('SERPAPI_API_KEY', '').strip() or getpass('SERPAPI_API_KEY: ').strip()
    if not API_KEY:
        raise ValueError('SERPAPI_API_KEY')

In [ ]:
def script1(output_dir, api_key='', fetch_abstracts=True):
    SERPAPI_KEY = api_key
    FETCH_ABSTRACTS = fetch_abstracts
    output_dir = Path(output_dir)
    errors = []
    REQUEST_DELAY = 1.2
    RESULTS_PER_PAGE = 100
    MAX_PAGES = 2
    BASELINE_TERMS = ['customs automation', 'manifest generation', 'automated classification customs', 'customs clearance system', 'duty calculation automated', 'tariff classification system']
    THRESHOLD_TERMS = ['value-based routing', 'threshold processing', 'low-value shipment', 'duty-free threshold', 'customs threshold automation', 'shipment value classification']
    CUSTOMS_CORE = ['customs', 'clearance', 'manifest', 'entry', 'declaration']
    CPC_PRECISION = ['G06F16/27', 'B07C', 'G06K7/10', 'G06Q10/08']
    NOISE_KEYWORDS = ['battery', 'semiconductor', 'display', 'antenna', 'pharma', 'pharmaceutical', 'lithium', 'drone', 'unmanned aerial', 'cryptocurrency', 'blockchain', 'solar panel', 'photovoltaic', 'medical device', 'diagnostic']

    def clean_text(text) -> str:
        if not isinstance(text, str):
            return ''
        replacements = {'â€¦': '…', 'â€"': '–', 'â€"': '—', 'â€™': "'", 'â€˜': "'", 'â€œ': '"', 'â€': '"', 'Ã©': 'é', 'Ã¨': 'è', 'Ã¼': 'ü', 'Ã¶': 'ö', 'Ã¤': 'ä', 'Ã±': 'ñ', 'Ã¡': 'á', 'Ã³': 'ó', 'Ã\xad': 'í', 'Ãº': 'ú', 'Â®': '®', 'Â©': '©', 'Â°': '°', 'Â': '', '&amp;': '&', '&lt;': '<', '&gt;': '>', '&quot;': '"'}
        for bad, good in replacements.items():
            text = text.replace(bad, good)
        return ' '.join(text.split())

    def _quote_assignee(a: str) -> str:
        return f'"{a}"' if ' ' in a and (not a.startswith('"')) else a

    def build_q(base_terms: str, assignee: Optional[str]=None, cpcs: Optional[List[str]]=None, after_year: int=2010, before_year: int=2026, require_customs: bool=True) -> str:
        parts = []
        if base_terms:
            if require_customs:
                customs_clause = '(' + ' OR '.join([f'"{c}"' for c in CUSTOMS_CORE]) + ')'
                parts.append(f'({customs_clause} AND ({base_terms}))')
            else:
                parts.append(f'({base_terms})')
        if assignee:
            parts.append(f'assignee:{_quote_assignee(assignee)}')
        if cpcs:
            cpc_clause = ' OR '.join(cpcs)
            parts.append(f'CPC:({cpc_clause})')
        parts.append(f'after:{after_year} before:{before_year}')
        return ' '.join(parts)

    def search_patents_serpapi(query_q: str, page: int=1) -> Optional[Dict]:
        import requests
        url = 'https://serpapi.com/search.json'
        params = {'engine': 'google_patents', 'q': query_q, 'api_key': SERPAPI_KEY, 'num': RESULTS_PER_PAGE}
        if page > 1:
            params['page'] = page
        try:
            r = requests.get(url, params=params, timeout=30)
            r.encoding = 'utf-8'
            r.raise_for_status()
            return r.json()
        except Exception as e:
            errors.append({'operation': 'search', 'identifier': query_q, 'error': type(e).__name__})
            return None

    def get_patent_details(patent_id: str) -> Dict:
        import requests
        url = 'https://serpapi.com/search.json'
        params = {'engine': 'google_patents_details', 'patent_id': patent_id, 'api_key': SERPAPI_KEY}
        try:
            r = requests.get(url, params=params, timeout=30)
            r.encoding = 'utf-8'
            r.raise_for_status()
            data = r.json()
            details = data.get('patent_details', {}) if isinstance(data, dict) else {}
            abstract = details.get('abstract', '')
            if not abstract and 'description' in details:
                desc = details.get('description', '')
                if isinstance(desc, str):
                    abstract = desc[:1500]
            return {'abstract': clean_text(abstract) if abstract else 'Not available', 'claims': clean_text(details.get('claims', '')), 'description': clean_text(details.get('description', '')[:2000])}
        except Exception as e:
            errors.append({'operation': 'details', 'identifier': patent_id, 'error': type(e).__name__})
            return {'abstract': 'Error fetching', 'claims': '', 'description': ''}

    def is_noise(title: str, snippet: str) -> bool:
        combined = (title + ' ' + snippet).lower()
        return any((noise_word in combined for noise_word in NOISE_KEYWORDS))

    def generate_baseline_threshold_queries() -> List[Dict]:
        queries = []
        top_carriers = ['United Parcel Service', 'Federal Express Corporation', 'DHL Express']
        for company in top_carriers:
            for term in BASELINE_TERMS[:3]:
                queries.append({'tier': 'A-Baseline', 'type': 'Carrier Infrastructure', 'assignee': company, 'cpcs': CPC_PRECISION, 'keyword': term, 'query': term, 'after': 2010, 'before': 2016, 'description': f'A-{company[:20]}: {term}'})
        top_customs = ['The Descartes Systems Group Inc.', 'Flexport Inc.', 'Livingston International Inc.']
        for company in top_customs:
            for term in BASELINE_TERMS[:4]:
                queries.append({'tier': 'A-Baseline', 'type': 'Customs Tech Infrastructure', 'assignee': company, 'cpcs': CPC_PRECISION, 'keyword': term, 'query': term, 'after': 2010, 'before': 2016, 'description': f'A-{company[:20]}: {term}'})
        for term in BASELINE_TERMS[:4]:
            queries.append({'tier': 'A-Baseline', 'type': 'General Infrastructure', 'assignee': None, 'cpcs': CPC_PRECISION, 'keyword': term, 'query': term, 'after': 2010, 'before': 2016, 'description': f'A-ALL: {term}'})
        all_companies = top_carriers + top_customs + ['Amazon Technologies, Inc.']
        for company in all_companies:
            for term in THRESHOLD_TERMS[:2]:
                queries.append({'tier': 'B-Threshold', 'type': 'Value-Based Processing', 'assignee': company, 'cpcs': CPC_PRECISION, 'keyword': term, 'query': term, 'after': 2016, 'before': 2026, 'description': f'B-{company[:20]}: {term}'})
        for term in THRESHOLD_TERMS:
            queries.append({'tier': 'B-Threshold', 'type': 'General Threshold Processing', 'assignee': None, 'cpcs': CPC_PRECISION, 'keyword': term, 'query': term, 'after': 2016, 'before': 2026, 'description': f'B-ALL: {term}'})
        return queries

    def main():
        output_dir.mkdir(parents=True, exist_ok=True)
        if SERPAPI_KEY in ('', 'YOUR_API_KEY_HERE'):
            raise ValueError('SERPAPI_API_KEY')
        queries = generate_baseline_threshold_queries()
        qrows = []
        for q in queries:
            qrows.append({**q, 'full_q': build_q(q['query'], assignee=q['assignee'], cpcs=q['cpcs'], after_year=q['after'], before_year=q['before'], require_customs=True)})
        qdf = pd.DataFrame(qrows)
        qdf.to_csv(output_dir / 'query_plan_script1.csv', index=False, encoding='utf-8-sig')
        all_patents = []
        api_calls = 0
        noise_filtered = 0
        for i, qi in enumerate(qrows, 1):
            q_strict = qi['full_q']
            got_results = False
            for page in range(1, MAX_PAGES + 1):
                data = search_patents_serpapi(q_strict, page=page)
                api_calls += 1
                if not data:
                    break
                org = data.get('organic_results', [])
                if org:
                    got_results = True
                    filtered = 0
                    for p in org:
                        title = clean_text(p.get('title', ''))
                        snippet = clean_text(p.get('snippet', ''))
                        if is_noise(title, snippet):
                            noise_filtered += 1
                            filtered += 1
                            continue
                        all_patents.append({'patent_id': p.get('patent_id', ''), 'title': title, 'snippet': snippet, 'assignee': clean_text(p.get('assignee', 'Unknown')), 'inventor': clean_text(p.get('inventor', '')), 'publication_date': p.get('publication_date', ''), 'filing_date': p.get('filing_date', ''), 'priority_date': p.get('priority_date', ''), 'grant_date': p.get('grant_date', ''), 'publication_number': p.get('publication_number', ''), 'patent_link': p.get('patent_link', ''), 'pdf': p.get('pdf', ''), 'tier': qi['tier'], 'type': qi['type'], 'search_assignee': qi['assignee'] or 'ALL', 'keyword': qi['keyword'], 'base_query': qi['query'], 'cpcs_used': str(qi['cpcs']), 'q_used': q_strict})
                if not org or len(org) < RESULTS_PER_PAGE:
                    break
                if page < MAX_PAGES:
                    time.sleep(REQUEST_DELAY)
            if not got_results and qi['cpcs']:
                q_fallback = build_q(qi['query'], assignee=qi['assignee'], cpcs=None, after_year=qi['after'], before_year=qi['before'], require_customs=True)
                for page in range(1, MAX_PAGES + 1):
                    data = search_patents_serpapi(q_fallback, page=page)
                    api_calls += 1
                    if not data:
                        break
                    org = data.get('organic_results', [])
                    if org:
                        filtered = 0
                        for p in org:
                            title = clean_text(p.get('title', ''))
                            snippet = clean_text(p.get('snippet', ''))
                            if is_noise(title, snippet):
                                noise_filtered += 1
                                filtered += 1
                                continue
                            all_patents.append({'patent_id': p.get('patent_id', ''), 'title': title, 'snippet': snippet, 'assignee': clean_text(p.get('assignee', 'Unknown')), 'inventor': clean_text(p.get('inventor', '')), 'publication_date': p.get('publication_date', ''), 'filing_date': p.get('filing_date', ''), 'priority_date': p.get('priority_date', ''), 'grant_date': p.get('grant_date', ''), 'publication_number': p.get('publication_number', ''), 'patent_link': p.get('patent_link', ''), 'pdf': p.get('pdf', ''), 'tier': qi['tier'], 'type': qi['type'], 'search_assignee': qi['assignee'] or 'ALL', 'keyword': qi['keyword'], 'base_query': qi['query'], 'cpcs_used': 'None (fallback)', 'q_used': q_fallback})
                    if not org or len(org) < RESULTS_PER_PAGE:
                        break
                    if page < MAX_PAGES:
                        time.sleep(REQUEST_DELAY)
            time.sleep(REQUEST_DELAY)
        if not all_patents:
            if errors:
                pd.DataFrame(errors).to_csv(output_dir / 'request_errors_script1.csv', index=False)
            raise RuntimeError('No patent records returned')
        df = pd.DataFrame(all_patents)
        initial = len(df)
        df = df.drop_duplicates(subset=['patent_id'], keep='first')
        final = len(df)
        df.to_csv(output_dir / 'patents_script1_raw.csv', index=False, encoding='utf-8-sig')
        if FETCH_ABSTRACTS and final > 0:
            abstracts, claims_list, descriptions = ([], [], [])
            for i, row in df.iterrows():
                details = get_patent_details(row['patent_id'])
                abstracts.append(details['abstract'])
                claims_list.append(details['claims'])
                descriptions.append(details['description'])
                api_calls += 1
                time.sleep(REQUEST_DELAY)
            df['abstract'] = abstracts
            df['claims'] = claims_list
            df['description_excerpt'] = descriptions
            df.to_csv(output_dir / 'patents_script1_with_abstracts.csv', index=False, encoding='utf-8-sig')
        if errors:
            pd.DataFrame(errors).to_csv(output_dir / 'request_errors_script1.csv', index=False)
        print({'script': 1, 'records': len(df), 'api_calls': api_calls, 'request_errors': len(errors)})
        return df
    queries = generate_baseline_threshold_queries()
    plan = pd.DataFrame([dict(q, full_q=build_q(q['query'], assignee=q['assignee'], cpcs=q['cpcs'], after_year=q['after'], before_year=q['before'], require_customs=True)) for q in queries])
    return (plan, main)

In [ ]:
def script2(output_dir, api_key='', fetch_abstracts=True):
    SERPAPI_KEY = api_key
    FETCH_ABSTRACTS = fetch_abstracts
    output_dir = Path(output_dir)
    errors = []
    REQUEST_DELAY = 1.2
    RESULTS_PER_PAGE = 100
    MAX_PAGES = 2
    AFTER_YEAR = 2016
    BEFORE_YEAR = 2026

    def clean_text(text) -> str:
        if not isinstance(text, str):
            return ''
        replacements = {'â€¦': '…', 'â€"': '–', 'â€"': '—', 'â€™': "'", 'â€˜': "'", 'â€œ': '"', 'â€': '"', 'Ã©': 'é', 'Ã¨': 'è', 'Ã¼': 'ü', 'Ã¶': 'ö', 'Ã¤': 'ä', 'Ã±': 'ñ', 'Ã¡': 'á', 'Ã³': 'ó', 'Ã\xad': 'í', 'Ãº': 'ú', 'Â®': '®', 'Â©': '©', 'Â°': '°', 'Â': '', '&amp;': '&', '&lt;': '<', '&gt;': '>', '&quot;': '"'}
        for bad, good in replacements.items():
            text = text.replace(bad, good)
        return ' '.join(text.split())
    TIER1_CARRIERS = ['FedEx', 'FedEx Corporation', 'Federal Express Corporation', 'UPS', 'United Parcel Service', 'United Parcel Service of America, Inc.', 'DHL', 'DHL Express', 'Deutsche Post', 'Deutsche Post AG']
    TIER2_CUSTOMS_TECH = ['Descartes', 'Descartes Systems Group', 'The Descartes Systems Group Inc.', 'Descartes Systems (USA) LLC', 'Flexport', 'Flexport Inc.', 'Flexport International LLC', 'Livingston', 'Livingston International', 'Livingston International Inc.']
    TIER3_PLATFORMS = ['Amazon', 'Amazon Technologies, Inc.', 'Amazon.com Services LLC', 'Shein', 'Roadget Business Pte. Ltd.', 'Zoetop Business Co., Ltd.', 'Temu', 'Whaleco Inc.', 'PDD Holdings Inc.', 'Alibaba', 'Alibaba Group Holding Limited', 'Alibaba (China) Co., Ltd.', 'Cainiao Smart Logistics Network Ltd.']
    SECTION321_DIRECT = ['"Section 321"', '"de minimis"', '"Entry Type 86"', '"low value shipment"', '"duty free threshold"', '"de minimis threshold"', '"de minimis exemption"']
    CUSTOMS_THRESHOLD = ['customs clearance automated threshold', 'customs clearance automated value', 'customs threshold processing', 'automated classification customs value', 'value-based routing customs', 'import compliance automated threshold']
    ECOMMERCE_CROSSBORDER = ['cross-border e-commerce customs', 'cross-border e-commerce import', 'cross-border e-commerce duty', 'international shipping fulfillment customs', 'direct-to-consumer customs import']
    GENERAL_LOGISTICS = ['parcel customs clearance', 'package customs automation', 'shipment customs threshold', 'logistics customs value']
    CPC_CORE = ['G06Q10/08', 'G06Q50/30', 'G06F16/27', 'G06K7/10', 'G06K9/00', 'B07C', 'B65G', 'B65D', 'G06Q30/06', 'G06Q20/00']

    def _quote_assignee(a: str) -> str:
        return a if a.startswith('"') or ' ' not in a else f'"{a}"'

    def build_q(base_terms: str, assignee: Optional[str]=None, cpcs: Optional[List[str]]=None) -> str:
        parts = []
        if base_terms:
            parts.append(f'({base_terms})')
        if assignee:
            parts.append(f'assignee:{_quote_assignee(assignee)}')
        if cpcs:
            parts.append('CPC:(' + ' OR '.join(cpcs) + ')')
        parts.append(f'after:{AFTER_YEAR} before:{BEFORE_YEAR}')
        return ' '.join(parts)

    def search_patents_serpapi(query_q: str, page: int=1) -> Optional[Dict]:
        import requests
        url = 'https://serpapi.com/search.json'
        params = {'engine': 'google_patents', 'q': query_q, 'api_key': SERPAPI_KEY, 'num': RESULTS_PER_PAGE}
        if page > 1:
            params['page'] = page
        try:
            r = requests.get(url, params=params, timeout=30)
            r.encoding = 'utf-8'
            r.raise_for_status()
            return r.json()
        except Exception as e:
            errors.append({'operation': 'search', 'identifier': query_q, 'error': type(e).__name__})
            return None

    def get_patent_details(patent_id: str) -> Dict:
        import requests
        url = 'https://serpapi.com/search.json'
        params = {'engine': 'google_patents_details', 'patent_id': patent_id, 'api_key': SERPAPI_KEY}
        try:
            r = requests.get(url, params=params, timeout=30)
            r.encoding = 'utf-8'
            r.raise_for_status()
            data = r.json()
            details = data.get('patent_details', {}) if isinstance(data, dict) else {}
            abstract = details.get('abstract') or (details.get('description', '')[:1000] if isinstance(details.get('description', ''), str) else 'Not available')
            return {'abstract': clean_text(abstract), 'claims': clean_text(details.get('claims', '')), 'description': clean_text(details.get('description', ''))}
        except Exception as e:
            errors.append({'operation': 'details', 'identifier': patent_id, 'error': type(e).__name__})
            return {'abstract': 'Error fetching', 'claims': '', 'description': ''}

    def generate_section321_queries() -> List[Dict]:
        queries = []
        for term in SECTION321_DIRECT:
            queries.append({'tier': 'Overarching', 'type': 'Section 321 Direct', 'assignee': None, 'cpcs': CPC_CORE, 'keyword': term, 'query': term, 'description': f'ALL: {term}'})
        combined_321 = ' OR '.join(SECTION321_DIRECT)
        queries.append({'tier': 'Overarching', 'type': 'Section 321 Combined', 'assignee': None, 'cpcs': CPC_CORE, 'keyword': 'All Section 321 terms', 'query': f'({combined_321})', 'description': 'ALL: Section 321/de minimis combined'})
        for company in TIER1_CARRIERS:
            for term in SECTION321_DIRECT[:3]:
                queries.append({'tier': 'Tier 1', 'type': 'Section 321 Primary', 'assignee': company, 'cpcs': CPC_CORE, 'keyword': term, 'query': term, 'description': f'T1-{company}: {term}'})
            for term in CUSTOMS_THRESHOLD[:2]:
                queries.append({'tier': 'Tier 1', 'type': 'Customs Threshold', 'assignee': company, 'cpcs': CPC_CORE, 'keyword': term, 'query': term, 'description': f'T1-{company}: {term}'})
        for company in TIER2_CUSTOMS_TECH:
            for term in CUSTOMS_THRESHOLD[:3]:
                queries.append({'tier': 'Tier 2', 'type': 'Threshold Processing', 'assignee': company, 'cpcs': CPC_CORE, 'keyword': term, 'query': term, 'description': f'T2-{company}: {term}'})
            queries.append({'tier': 'Tier 2', 'type': 'Automated Entry', 'assignee': company, 'cpcs': CPC_CORE, 'keyword': 'customs entry automated', 'query': 'customs entry automated', 'description': f'T2-{company}: customs entry automated'})
        for company in TIER3_PLATFORMS:
            for term in ECOMMERCE_CROSSBORDER[:2]:
                queries.append({'tier': 'Tier 3', 'type': 'E-commerce Cross-border', 'assignee': company, 'cpcs': CPC_CORE, 'keyword': term, 'query': term, 'description': f'T3-{company}: {term}'})
            queries.append({'tier': 'Tier 3', 'type': 'International Fulfillment', 'assignee': company, 'cpcs': CPC_CORE, 'keyword': 'international shipping fulfillment', 'query': 'international shipping fulfillment', 'description': f"T3-{company}: int'l shipping"})
        for term in GENERAL_LOGISTICS:
            queries.append({'tier': 'General', 'type': 'Logistics Context', 'assignee': None, 'cpcs': None, 'keyword': term, 'query': term, 'description': f'General: {term}'})
        for cpc in CPC_CORE[:5]:
            queries.append({'tier': 'CPC Technical', 'type': 'CPC + Section 321', 'assignee': None, 'cpcs': [cpc], 'keyword': f'{cpc} + Section 321', 'query': '"Section 321"', 'description': f'CPC {cpc}: Section 321'})
            queries.append({'tier': 'CPC Technical', 'type': 'CPC + de minimis', 'assignee': None, 'cpcs': [cpc], 'keyword': f'{cpc} + de minimis', 'query': '"de minimis"', 'description': f'CPC {cpc}: de minimis'})
        return queries

    def main():
        output_dir.mkdir(parents=True, exist_ok=True)
        if SERPAPI_KEY in ('', 'YOUR_API_KEY_HERE'):
            raise ValueError('SERPAPI_API_KEY')
        queries = generate_section321_queries()
        qrows = []
        for q in queries:
            qrows.append({**q, 'full_q': build_q(q['query'], assignee=q['assignee'], cpcs=q['cpcs'])})
        qdf = pd.DataFrame(qrows)
        qdf.to_csv(output_dir / 'query_plan_script2.csv', index=False, encoding='utf-8-sig')
        all_patents, api_calls = ([], 0)
        for i, qi in enumerate(qrows, 1):
            q_strict = qi['full_q']
            got = 0
            for page in range(1, MAX_PAGES + 1):
                data = search_patents_serpapi(q_strict, page=page)
                api_calls += 1
                time.sleep(REQUEST_DELAY)
                org = data.get('organic_results', []) if data else []
                got += len(org)
                for p in org:
                    all_patents.append({'patent_id': p.get('patent_id', ''), 'title': clean_text(p.get('title', '')), 'snippet': clean_text(p.get('snippet', '')), 'assignee': clean_text(p.get('assignee', 'Unknown')), 'inventor': clean_text(p.get('inventor', '')), 'publication_date': p.get('publication_date', ''), 'filing_date': p.get('filing_date', ''), 'priority_date': p.get('priority_date', ''), 'grant_date': p.get('grant_date', ''), 'publication_number': p.get('publication_number', ''), 'language': p.get('language', ''), 'patent_link': p.get('patent_link', ''), 'pdf': p.get('pdf', ''), 'tier': qi['tier'], 'type': qi['type'], 'search_assignee': qi['assignee'] or 'ALL', 'keyword': qi['keyword'], 'base_query': qi['query'], 'cpcs_used': str(qi['cpcs']) if qi['cpcs'] else 'None', 'q_used': q_strict})
                if not org or len(org) < RESULTS_PER_PAGE:
                    break
            if got == 0 and qi['cpcs']:
                q_fallback = build_q(qi['query'], assignee=qi['assignee'], cpcs=None)
                for page in range(1, MAX_PAGES + 1):
                    data = search_patents_serpapi(q_fallback, page=page)
                    api_calls += 1
                    time.sleep(REQUEST_DELAY)
                    org = data.get('organic_results', []) if data else []
                    for p in org:
                        all_patents.append({'patent_id': p.get('patent_id', ''), 'title': clean_text(p.get('title', '')), 'snippet': clean_text(p.get('snippet', '')), 'assignee': clean_text(p.get('assignee', 'Unknown')), 'inventor': clean_text(p.get('inventor', '')), 'publication_date': p.get('publication_date', ''), 'filing_date': p.get('filing_date', ''), 'priority_date': p.get('priority_date', ''), 'grant_date': p.get('grant_date', ''), 'publication_number': p.get('publication_number', ''), 'language': p.get('language', ''), 'patent_link': p.get('patent_link', ''), 'pdf': p.get('pdf', ''), 'tier': qi['tier'], 'type': qi['type'], 'search_assignee': qi['assignee'] or 'ALL', 'keyword': qi['keyword'], 'base_query': qi['query'], 'cpcs_used': 'None (fallback)', 'q_used': q_fallback})
                    if not org or len(org) < RESULTS_PER_PAGE:
                        break
        if not all_patents:
            if errors:
                pd.DataFrame(errors).to_csv(output_dir / 'request_errors_script2.csv', index=False)
            raise RuntimeError('No patent records returned')
        df = pd.DataFrame(all_patents).drop_duplicates(subset=['patent_id'])
        df.to_csv(output_dir / 'patents_script2_raw.csv', index=False, encoding='utf-8-sig')
        if FETCH_ABSTRACTS and (not df.empty):
            abstracts, claims_list, descriptions = ([], [], [])
            for i, row in df.iterrows():
                d = get_patent_details(row['patent_id'])
                time.sleep(REQUEST_DELAY)
                abstracts.append(d['abstract'])
                claims_list.append(d['claims'])
                descriptions.append(d['description'])
                api_calls += 1
            df['abstract'] = abstracts
            df['claims'] = claims_list
            df['description_excerpt'] = descriptions
            df.to_csv(output_dir / 'patents_script2_with_abstracts.csv', index=False, encoding='utf-8-sig')
        if errors:
            pd.DataFrame(errors).to_csv(output_dir / 'request_errors_script2.csv', index=False)
        print({'script': 2, 'records': len(df), 'api_calls': api_calls, 'request_errors': len(errors)})
        return df
    queries = generate_section321_queries()
    plan = pd.DataFrame([dict(q, full_q=build_q(q['query'], assignee=q['assignee'], cpcs=q['cpcs'])) for q in queries])
    return (plan, main)

In [ ]:
plan1, collect1 = script1(COLLECTION_DIR, API_KEY, FETCH_ABSTRACTS)
plan2, collect2 = script2(COLLECTION_DIR, API_KEY, FETCH_ABSTRACTS)
plans = [plan1, plan2]
for number, plan in enumerate(plans, 1):
    plan.to_csv(RESULTS_DIR / f'query_plan_script{number}.csv', index=False)
if RUN_COLLECTION:
    frames = [collect1(), collect2()]
else:
    frames = [pd.read_csv(DATA_DIR / name, dtype=str, keep_default_na=False) for name in DATA_FILES]
required = {'patent_id', 'publication_number', 'title', 'snippet', 'assignee', 'type', 'keyword'}
for number, frame in enumerate(frames, 1):
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f'script{number}: {sorted(missing)}')
    if frame['publication_number'].eq('').any():
        raise ValueError(f'script{number}: publication_number')
    if frame['patent_id'].duplicated().any():
        raise ValueError(f'script{number}: duplicate patent_id')
    if frame['publication_number'].duplicated().any():
        raise ValueError(f'script{number}: duplicate publication_number')
    text_columns = [column for column in ['title', 'snippet', 'abstract'] if column in frame]
    frame['_text'] = frame[text_columns].fillna('').astype(str).agg(' '.join, axis=1).str.lower()
    frame['source_script'] = number
df1, df2 = frames

In [ ]:
AMAZON_ALIASES = {'アマゾン・テクノロジーズ・インコーポレーテッド', '亚马逊科技公司'}
def amazon_publications(frame):
    mask = frame['assignee'].str.contains('amazon', case=False, regex=False, na=False)
    mask = mask | frame['assignee'].isin(AMAZON_ALIASES)
    return set(frame.loc[mask, 'publication_number'])

ids1 = set(df1['publication_number'])
ids2 = set(df2['publication_number'])
amazon1 = amazon_publications(df1)
amazon2 = amazon_publications(df2)
direct = df2.loc[df2['type'].eq('Section 321 Direct')].copy()
counts = pd.DataFrame([
    ('script1_queries', len(plan1)),
    ('script2_queries', len(plan2)),
    ('total_queries', len(plan1) + len(plan2)),
    ('script1_records', len(df1)),
    ('script2_records', len(df2)),
    ('total_records', len(df1) + len(df2)),
    ('overlapping_publications', len(ids1 & ids2)),
    ('unique_publications', len(ids1 | ids2)),
    ('amazon_script1', len(amazon1)),
    ('amazon_script2', len(amazon2)),
    ('amazon_overlap', len(amazon1 & amazon2)),
    ('amazon_unique', len(amazon1 | amazon2)),
    ('direct_search_publications', direct['publication_number'].nunique())
], columns=['measure', 'value'])
counts.to_csv(RESULTS_DIR / 'patent_counts.csv', index=False)
combined = pd.concat(frames, ignore_index=True)
unique = combined.drop_duplicates('publication_number', keep='first').drop(columns='_text')
unique.to_csv(RESULTS_DIR / 'patents_unique.csv', index=False)
direct.drop(columns='_text').to_csv(RESULTS_DIR / 'direct_search_records.csv', index=False)
amazon = unique.loc[unique['publication_number'].isin(amazon1 | amazon2)]
amazon.to_csv(RESULTS_DIR / 'amazon_patents.csv', index=False)
print(counts.to_string(index=False))

In [ ]:
KEYWORDS = [
    r'\$800',
    r'de.?minimis',
    r'section 321',
    r'entry type 86',
    r'duty.?free threshold',
    r'low.?value shipment',
    r'tariff exempt',
    r'customs (threshold|exempt)',
    r'import threshold'
]
LITERAL_TERMS = [
    '$800', 'de minimis', 'de-minimis', 'deminimis', 'section 321',
    'entry type 86', 'duty-free threshold', 'customs threshold',
    'value threshold', 'low-value shipment', 'tariff exempt',
    'duty exempt', 'customs exempt', 'import threshold', 'clearance threshold'
]
keyword_counts = []
keyword_hits = []
for number, frame in enumerate(frames, 1):
    for method, terms in [('regex', KEYWORDS), ('literal', LITERAL_TERMS)]:
        for term in terms:
            if method == 'regex':
                pattern = re.compile(term, re.IGNORECASE)
                mask = frame['_text'].map(lambda text: bool(pattern.search(text)))
            else:
                mask = frame['_text'].str.contains(term, regex=False, case=False, na=False)
            keyword_counts.append((number, method, term, int(mask.sum())))
            for publication in frame.loc[mask, 'publication_number']:
                keyword_hits.append((number, method, term, publication))
keyword_counts = pd.DataFrame(keyword_counts, columns=['source_script', 'method', 'term', 'hits'])
keyword_hits = pd.DataFrame(keyword_hits, columns=['source_script', 'method', 'term', 'publication_number'])
keyword_counts.to_csv(RESULTS_DIR / 'keyword_counts.csv', index=False)
keyword_hits.to_csv(RESULTS_DIR / 'keyword_hits.csv', index=False)
print(keyword_hits.to_string(index=False))